# 03 — LightGBM quantile regression at τ=0.2
Does not beat the per-rm linear (Group 74's experience confirms). Kept here as a candidate for ensemble weight 0.0–0.2.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd().parent))
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
from src.data import load_or_build, build_profile
from src.features import build_features
from src.models.lgbm_quantile import LGBMQuantileForecaster, assemble_training_set
from src.validation import DEFAULT_FOLDS, evaluate, build_query_for_fold


In [ ]:
ds = load_or_build()
rm_ids = sorted(ds.daily['rm_id'].unique().tolist())
for fold in DEFAULT_FOLDS:
    cutoff = fold.train_end + pd.Timedelta(days=1)
    daily_pre = ds.daily[ds.daily['date'] < cutoff]
    profile = build_profile(daily_pre, year=fold.target_year - 1)
    train_years = [y for y in [2020, 2021, 2022, 2023] if y < fold.target_year]
    base_end = pd.date_range('2020-01-02', '2020-05-31', freq='D')
    X, y, w = assemble_training_set(daily_pre, ds.materials, train_years, base_end, rm_ids, profile_for_weight=profile)
    fold_dates = pd.date_range(f'{fold.target_year}-01-02', f'{fold.target_year}-05-31', freq='D')
    val = build_features(daily_pre, ds.materials, fold.target_year, fold_dates, rm_ids)
    truth = build_features(ds.daily, ds.materials, fold.target_year, fold_dates, rm_ids).target
    m = LGBMQuantileForecaster()
    m.fit(X, y, valid_df=val.features, valid_target=truth, sample_weight=w)
    preds = m.predict(val.features)
    s = evaluate(preds, fold, ds.daily)
    print(f'{fold.name} pinball={s["mean_pinball"]:.0f}  best_iter={m.booster.best_iteration}')
